# Output Visual dan Tabel BAB 4

Notebook ini membuat artefak tabel dan visual untuk BAB 4 berdasarkan `TODO_BAB_4_Lengkap.md`.
Kode subbab 4.1.1 sudah dipisahkan ke `notebooks/generate_bab4_4_1_1_outputs.ipynb`, kode subbab 4.1.2 sudah dipisahkan ke `notebooks/generate_bab4_4_1_2_outputs.ipynb`, kode subbab 4.2 sudah dipisahkan ke `notebooks/generate_bab4_4_2_outputs.ipynb`, kode subbab 4.3 sudah dipisahkan ke `notebooks/generate_bab4_4_3_outputs.ipynb`, kode subbab 4.4.1 sudah dipisahkan ke `notebooks/generate_bab4_4_4_1_outputs.ipynb`, kode subbab 4.4.2 sudah dipisahkan ke `notebooks/generate_bab4_4_4_2_outputs.ipynb`, kode subbab 4.4.3 sudah dipisahkan ke `notebooks/generate_bab4_4_4_3_outputs.ipynb`, kode subbab 4.5 sudah dipisahkan ke `notebooks/generate_bab4_4_5_outputs.ipynb`, kode subbab 4.6 sudah dipisahkan ke `notebooks/generate_bab4_4_6_outputs.ipynb`, kode subbab 4.7 sudah dipisahkan ke `notebooks/generate_bab4_4_7_outputs.ipynb`, dan kode subbab 4.8 sudah dipisahkan ke `notebooks/generate_bab4_4_8_outputs.ipynb`; notebook ini melanjutkan output 4.9 dan seterusnya.


## 0. Setup

Konfigurasi path, folder output, plotting style, dan helper toleran. Jika artefak belum tersedia, notebook membuat tabel status/warning dan tetap lanjut.


In [1]:
from __future__ import annotations

import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import rasterio
except Exception as exc:
    rasterio = None
    RASTERIO_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"
else:
    RASTERIO_IMPORT_ERROR = ""

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "font.size": 10,
})

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATASET = ROOT / "dataset"
RUNS = ROOT / "runs"
OUT = ROOT / "outputs" / "bab4"
TABLE_DIR = OUT / "tables"
FIG_DIR = OUT / "figures"
NARRATIVE_DIR = OUT / "narratives"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
NARRATIVE_DIR.mkdir(parents=True, exist_ok=True)

REGIONS = [
    "Aceh_Besar", "Aceh_Tamiang", "Aceh_Timur", "Aceh_Utara", "Agam",
    "Banda_Aceh", "Bireuen", "Langsa", "Pasaman_Barat", "Pidie", "Pidie_Jaya",
]
TEST_REGION = "Aceh_Utara"
CHANNEL_NAMES = ["vv_norm", "vh_norm", "hue", "saturation", "value", "slope_norm", "hand_norm"]

print(f"ROOT: {ROOT}")
print(f"Tables: {TABLE_DIR}")
print(f"Figures: {FIG_DIR}")
print("rasterio:", "available" if rasterio else f"missing ({RASTERIO_IMPORT_ERROR})")


ROOT: /home/nozomi/Productive/skripsi
Tables: /home/nozomi/Productive/skripsi/outputs/bab4/tables
Figures: /home/nozomi/Productive/skripsi/outputs/bab4/figures
rasterio: available


In [2]:
def display_df(df: pd.DataFrame, max_rows: int = 20):
    if df.empty:
        display(df)
    else:
        display(df.head(max_rows))


def save_table(df: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print(f"saved table: {path.relative_to(ROOT)} ({len(df)} rows)")
    display_df(df)
    return path


def save_fig(fig, name: str) -> Path:
    path = FIG_DIR / name
    try:
        fig.tight_layout(rect=[0, 0, 1, 0.96])
    except Exception:
        fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    print(f"saved figure: {path.relative_to(ROOT)}")
    plt.show()
    return path


def read_csv_or_status(path: Path, label: str) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame([{"source": str(path.relative_to(ROOT)), "status": "missing", "note": label}])
    try:
        return pd.read_csv(path)
    except Exception as exc:
        return pd.DataFrame([{"source": str(path.relative_to(ROOT)), "status": "read_error", "note": f"{type(exc).__name__}: {exc}"}])


def safe_div(num: float, den: float) -> float:
    return float(num / den) if den else 0.0


def pct(num: float, den: float) -> float:
    return round(100.0 * safe_div(num, den), 4)


def finite_stats(arr: np.ndarray, sample_step: int = 1) -> dict:
    a = np.asarray(arr)
    if sample_step > 1 and a.ndim >= 2:
        a = a[..., ::sample_step, ::sample_step]
    a = a.astype("float64", copy=False)
    finite = np.isfinite(a)
    vals = a[finite]
    if vals.size == 0:
        return {"count": 0, "valid_pct": 0.0, "min": np.nan, "max": np.nan, "mean": np.nan, "std": np.nan, "p2": np.nan, "p98": np.nan}
    return {
        "count": int(vals.size),
        "valid_pct": round(100.0 * vals.size / a.size, 4),
        "min": float(np.min(vals)),
        "max": float(np.max(vals)),
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals)),
        "p2": float(np.percentile(vals, 2)),
        "p98": float(np.percentile(vals, 98)),
    }


def normalize_image(arr: np.ndarray, p_low: float = 2, p_high: float = 98) -> np.ndarray:
    a = np.asarray(arr, dtype="float32")
    finite = np.isfinite(a)
    if not finite.any():
        return np.zeros_like(a, dtype="float32")
    lo, hi = np.percentile(a[finite], [p_low, p_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.nanmin(a[finite])), float(np.nanmax(a[finite]))
    if hi <= lo:
        return np.zeros_like(a, dtype="float32")
    return np.clip((a - lo) / (hi - lo), 0, 1)


def raster_stats(path: Path, band: int = 1, sample_step: int = 10) -> dict:
    if rasterio is None:
        return {"status": "skipped", "note": f"rasterio unavailable: {RASTERIO_IMPORT_ERROR}"}
    if not path.exists():
        return {"status": "missing", "note": str(path.relative_to(ROOT))}
    try:
        with rasterio.open(path) as src:
            arr = src.read(band, masked=True)
            vals = arr.compressed() if np.ma.isMaskedArray(arr) else np.asarray(arr).ravel()
            if sample_step > 1:
                arr2 = np.asarray(arr.filled(np.nan) if np.ma.isMaskedArray(arr) else arr)
                vals = arr2[::sample_step, ::sample_step].ravel()
            stats = finite_stats(vals)
            stats.update({
                "status": "ok",
                "height": src.height,
                "width": src.width,
                "crs": str(src.crs),
                "transform": str(src.transform),
            })
            return stats
    except Exception as exc:
        return {"status": "read_error", "note": f"{type(exc).__name__}: {exc}"}


def load_npz(path: Path):
    return np.load(path, allow_pickle=False)


def first_npz(path: Path) -> Path | None:
    files = sorted(path.glob("*.npz"))
    return files[0] if files else None


def parse_tile_rc(path: Path) -> tuple[int | None, int | None]:
    m = re.search(r"_r(\d+)_c(\d+)", path.stem)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))


def choose_tile(region: str = TEST_REGION) -> Path | None:
    tile_dir = DATASET / "tiles" / "7ch" / "by_region" / region
    candidates = []
    for path in sorted(tile_dir.glob("*.npz")):
        try:
            z = load_npz(path)
            y = np.asarray(z["y"])[0]
            valid = np.asarray(z.get("valid_mask", np.ones_like(y)))[0]
            positives = int(((y > 0) & (valid > 0)).sum())
            valid_count = int((valid > 0).sum())
            candidates.append((positives, valid_count, path))
        except Exception:
            continue
    if not candidates:
        return None
    positives = [c for c in candidates if c[0] > 0]
    if positives:
        return sorted(positives, key=lambda x: x[0], reverse=True)[0][2]
    return sorted(candidates, key=lambda x: x[1], reverse=True)[0][2]


def find_matching_prediction(model: str, tile_path: Path) -> Path | None:
    pred_dir = RUNS / "final" / model / "eval_test" / "predictions" / TEST_REGION
    candidate = pred_dir / tile_path.name
    if candidate.exists():
        return candidate
    r, c = parse_tile_rc(tile_path)
    if r is None:
        return first_npz(pred_dir)
    matches = sorted(pred_dir.glob(f"*r{r:06d}_c{c:06d}.npz"))
    return matches[0] if matches else first_npz(pred_dir)


## 4.1.1 Statistik Hasil Preprocessing

Kode untuk subbab 4.1.1 sudah dipisahkan ke `notebooks/generate_bab4_4_1_1_outputs.ipynb`. Jalankan notebook tersebut untuk membuat statistik VV/VH, HSV, Slope, HAND, tabel validitas Sentinel-2, visual channel, raw dB/clipping Sentinel-1, dan narasi interpretasi. Notebook utama ini melanjutkan output BAB 4 mulai dari 4.1.2 dan tetap memeriksa keberadaan artefak 4.1.1 pada checklist akhir.

## 4.1.2 Verifikasi Alignment Raster Multisensor

Kode dan output subbab 4.1.2 sudah dipisahkan ke `notebooks/generate_bab4_4_1_2_outputs.ipynb`. Artefak yang dihasilkan tetap berada di `outputs/bab4/tables`, `outputs/bab4/figures`, dan `outputs/bab4/narratives` agar dapat dipakai oleh BAB 4 utama.


## 4.2 Hasil Pembentukan Label UNOSAT dan Dataset Tile

Kode dan output subbab 4.2 sudah dipisahkan ke `notebooks/generate_bab4_4_2_outputs.ipynb`. Artefak yang dihasilkan tetap berada di `outputs/bab4/tables`, `outputs/bab4/figures`, dan `outputs/bab4/narratives` agar dapat dipakai oleh BAB 4 utama.


## 4.3 Pembagian Dataset Spasial dan Validasi Eksperimental

Kode dan output subbab 4.3 sudah dipisahkan ke `notebooks/generate_bab4_4_3_outputs.ipynb`. Artefak yang dihasilkan tetap berada di `outputs/bab4/tables`, `outputs/bab4/figures`, dan `outputs/bab4/narratives` agar dapat dipakai oleh BAB 4 utama.


## 4.4.1 Hasil Implementasi Arsitektur Model

Kode dan output subbab 4.4.1 sudah dipisahkan ke `notebooks/generate_bab4_4_4_1_outputs.ipynb`. Artefak yang dihasilkan tetap berada di `outputs/bab4/tables`, `outputs/bab4/figures`, dan `outputs/bab4/narratives` agar dapat dipakai oleh BAB 4 utama.


## 4.4.2 Hasil Tuning Hyperparameter

Kode dan output subbab 4.4.2 sudah dipisahkan ke `notebooks/generate_bab4_4_4_2_outputs.ipynb`. Artefak yang dihasilkan tetap berada di `outputs/bab4/tables`, `outputs/bab4/figures`, dan `outputs/bab4/narratives` agar dapat dipakai oleh BAB 4 utama.


## 4.4.3 Stabilitas Pelatihan

Kode dan output subbab 4.4.3 sudah dipisahkan ke `notebooks/generate_bab4_4_4_3_outputs.ipynb`. Artefak yang dihasilkan tetap berada di `outputs/bab4/tables`, `outputs/bab4/figures`, dan `outputs/bab4/narratives` agar dapat dipakai oleh BAB 4 utama.


## 4.5 Evaluasi Akhir pada Wilayah Uji Aceh_Utara

Kode dan output subbab 4.5 sudah dipisahkan ke `notebooks/generate_bab4_4_5_outputs.ipynb`. Artefak yang dihasilkan tetap berada di `outputs/bab4/tables`, `outputs/bab4/figures`, dan `outputs/bab4/narratives` agar dapat dipakai oleh BAB 4 utama.


## 4.6 Analisis Visual dan Spasial Hasil Segmentasi

Kode dan output subbab 4.6 sudah dipisahkan ke `notebooks/generate_bab4_4_6_outputs.ipynb`. Artefak yang dihasilkan tetap berada di `outputs/bab4/tables`, `outputs/bab4/figures`, dan `outputs/bab4/narratives` agar dapat dipakai oleh BAB 4 utama.


## 4.7 Pembahasan Efektivitas U-Net vs ProCANet

Kode dan output subbab 4.7 sudah dipisahkan ke `notebooks/generate_bab4_4_7_outputs.ipynb`. Artefak yang dihasilkan tetap berada di `outputs/bab4/tables`, `outputs/bab4/figures`, dan `outputs/bab4/narratives` agar dapat dipakai oleh BAB 4 utama.


## 4.8 Ketahanan Model pada Kondisi Data Sulit

Kode dan output subbab 4.8 sudah dipisahkan ke `notebooks/generate_bab4_4_8_outputs.ipynb`. Artefak yang dihasilkan tetap berada di `outputs/bab4/tables`, `outputs/bab4/figures`, dan `outputs/bab4/narratives` agar dapat dipakai oleh BAB 4 utama.


## 4.9 Ringkasan Temuan Bab 4

Tabel mini untuk membantu narasi sintesis akhir BAB 4.


In [20]:
summary_rows = [
    {"subbab": "4.1", "tema": "Kualitas input multisensor", "temuan_ringkas": "SAR tersedia stabil; validitas Sentinel-2 bervariasi; DEMNAS memberi konteks topografi.", "output_pendukung": "4_1_1_preprocessing_stats.csv; 4_1_2_alignment_verification.csv"},
    {"subbab": "4.2", "tema": "Label dan class imbalance", "temuan_ringkas": "Label banjir minor dibanding non-banjir; valid mask membatasi area evaluasi.", "output_pendukung": "4_2_label_mask_tile_stats.csv"},
    {"subbab": "4.3", "tema": "Validasi spasial", "temuan_ringkas": "Split berbasis wilayah mengurangi kebocoran spasial antara training dan pengujian final.", "output_pendukung": "4_3_spatial_split_summary.csv"},
    {"subbab": "4.4", "tema": "Implementasi dan training", "temuan_ringkas": "U-Net menjadi baseline 7-channel; ProCANet memakai fusi dual-encoder/cross-attention.", "output_pendukung": "4_4_1_model_architecture_specs.csv; 4_4_3_training_curves.png"},
    {"subbab": "4.5-4.7", "tema": "Evaluasi model", "temuan_ringkas": "Perbandingan IoU, Dice, accuracy, FP, dan FN menunjukkan trade-off cakupan prediksi.", "output_pendukung": "4_5_final_metrics.csv; 4_7_unet_vs_procanet_effectiveness_summary.csv"},
    {"subbab": "4.8", "tema": "Ketahanan data sulit", "temuan_ringkas": "Kasus Sentinel-2 terbatas dan mask air/sungai perlu dibahas sebagai batasan interpretasi.", "output_pendukung": "4_8_difficult_data_case_studies.csv"},
]
chapter_summary = pd.DataFrame(summary_rows)
save_table(chapter_summary, "4_9_bab4_findings_summary.csv")


saved table: outputs/bab4/tables/4_9_bab4_findings_summary.csv (6 rows)


,subbab,tema,temuan_ringkas,output_pendukung
0,4.1,Kualitas input multisensor,SAR tersedia stabil; validitas Sentinel-2 berv...,4_1_1_preprocessing_stats.csv; 4_1_2_alignment...
1,4.2,Label dan class imbalance,Label banjir minor dibanding non-banjir; valid...,4_2_label_mask_tile_stats.csv
2,4.3,Validasi spasial,Split berbasis wilayah mengurangi kebocoran sp...,4_3_spatial_split_summary.csv
3,4.4,Implementasi dan training,U-Net menjadi baseline 7-channel; ProCANet mem...,4_4_1_model_architecture_specs.csv; 4_4_3_trai...
4,4.5-4.7,Evaluasi model,"Perbandingan IoU, Dice, accuracy, FP, dan FN m...",4_5_final_metrics.csv; 4_7_unet_vs_procanet_ef...
5,4.8,Ketahanan data sulit,Kasus Sentinel-2 terbatas dan mask air/sungai ...,4_8_difficult_data_case_studies.csv


PosixPath('/home/nozomi/Productive/skripsi/outputs/bab4/tables/4_9_bab4_findings_summary.csv')

## 4.7-4.9 Tambahan: Narasi Siap Pakai dan Coverage Matrix

Menyediakan tabel interpretasi/narasi untuk pembahasan serta matriks cakupan line 34-573.


In [27]:
narrative_rows = [
    {"subbab": "4.1", "narasi_siap_pakai": "Preprocessing menyatukan Sentinel-1, Sentinel-2 HSV, Slope, dan HAND ke grid Sentinel-1; statistik dan verifikasi stack menunjukkan input layak dipakai sebagai tensor multisensor."},
    {"subbab": "4.2", "narasi_siap_pakai": "Label positif berasal dari FloodExtent, sedangkan AnalysisExtent membatasi area valid dan WaterExtent/River dipakai sebagai konteks/mask air permanen."},
    {"subbab": "4.3", "narasi_siap_pakai": "Validasi berbasis wilayah memisahkan area tuning dari Aceh_Utara sebagai uji final sehingga evaluasi lebih tahan terhadap kebocoran spasial."},
    {"subbab": "4.4", "narasi_siap_pakai": "U-Net dan ProCANet lolos kontrak forward pass; training memakai AdamW, scheduler ReduceLROnPlateau, early stopping, dan checkpoint terbaik berbasis validation IoU."},
    {"subbab": "4.5-4.7", "narasi_siap_pakai": "Perbandingan akhir perlu dibaca sebagai trade-off: IoU/Dice mengukur overlap, sedangkan FP/FN menunjukkan kecenderungan overprediction atau underprediction."},
    {"subbab": "4.8", "narasi_siap_pakai": "Wilayah dengan Sentinel-2 terbatas, water/river mask besar, atau label banjir kecil menjadi studi kasus ketahanan data."},
    {"subbab": "4.9", "narasi_siap_pakai": "Temuan BAB 4 menjadi dasar BAB 5: kualitas input, validitas label, konfigurasi model, dan trade-off evaluasi akhir."},
]
save_table(pd.DataFrame(narrative_rows), "4_7_4_9_discussion_narrative_snippets.csv")

forward_ok = False
try:
    forward_ok = (forward_pass["status"] == "ok").all()
except Exception:
    forward_ok = False
osm_ok = (FIG_DIR / "4_1_2_osm_overlay_stack_aceh_utara.png").exists()

coverage_items = [
    ("4.1.1", "raw Sentinel-1 dB, clipping [-30,0], normalization [0,1]", "4_1_1_raw_s1_db_clipping_stats.csv", True),
    ("4.1.1", "normalized VV/VH/HSV/Slope/HAND stats and S2 valid mask", "4_1_1_preprocessing_stats.csv", True),
    ("4.1.2", "raster metadata/alignment", "4_1_2_alignment_verification.csv", True),
    ("4.1.2", "stack_7ch band verification", "4_1_2_stack_7ch_layer_verification.csv", True),
    ("4.1.2", "OSM overlay against stack layers", "4_1_2_osm_overlay_stack_aceh_utara.png", osm_ok),
    ("4.1.2", "same-location multi-layer tile panel", "4_1_2_same_tile_multilayer_panel_aceh_utara.png", (FIG_DIR / "4_1_2_same_tile_multilayer_panel_aceh_utara.png").exists()),
    ("4.1.2", "alignment interpretation narrative", "4_1_2_alignment_interpretation.md", (NARRATIVE_DIR / "4_1_2_alignment_interpretation.md").exists()),
    ("4.2", "UNOSAT source layers and label/mask stats", "4_2_unosat_source_layer_mapping.csv; 4_2_label_mask_tile_stats.csv", (TABLE_DIR / "4_2_unosat_source_layer_mapping.csv").exists() and (TABLE_DIR / "4_2_label_mask_tile_stats.csv").exists()),
    ("4.2", "UNOSAT mask panel", "4_2_unosat_mask_panel_aceh_utara.png", (FIG_DIR / "4_2_unosat_mask_panel_aceh_utara.png").exists()),
    ("4.2", "label/tile interpretation narrative", "4_2_label_tile_interpretation.md", (NARRATIVE_DIR / "4_2_label_tile_interpretation.md").exists()),
    ("4.3", "5-fold spatial cross-validation", "4_3_five_fold_spatial_cv.csv", (TABLE_DIR / "4_3_five_fold_spatial_cv.csv").exists()),
    ("4.3", "fold tile counts train/validation/final test", "4_3_fold_tile_counts.csv", (TABLE_DIR / "4_3_fold_tile_counts.csv").exists()),
    ("4.3", "spatial split diagram", "4_3_spatial_split_diagram.png", (FIG_DIR / "4_3_spatial_split_diagram.png").exists()),
    ("4.3", "spatial leakage narrative", "4_3_spatial_split_interpretation.md", (NARRATIVE_DIR / "4_3_spatial_split_interpretation.md").exists()),
    ("4.4.1", "architecture specs and forward pass", "4_4_1_model_architecture_specs.csv; 4_4_1_forward_pass_verification.csv", (TABLE_DIR / "4_4_1_model_architecture_specs.csv").exists() and (TABLE_DIR / "4_4_1_forward_pass_verification.csv").exists()),
    ("4.4.1", "actual architecture diagrams", "4_4_1_unet_architecture_diagram.png; 4_4_1_procanet_architecture_diagram.png", (FIG_DIR / "4_4_1_unet_architecture_diagram.png").exists() and (FIG_DIR / "4_4_1_procanet_architecture_diagram.png").exists()),
    ("4.4.1", "logit output and threshold narrative", "4_4_1_architecture_interpretation.md", (NARRATIVE_DIR / "4_4_1_architecture_interpretation.md").exists()),
    ("4.4.2", "hyperparameter grid search", "4_4_2_hyperparameter_tuning_summary.csv", (TABLE_DIR / "4_4_2_hyperparameter_tuning_summary.csv").exists()),
    ("4.4.2", "best hyperparameter table", "4_4_2_best_hyperparameters_by_model.csv", (TABLE_DIR / "4_4_2_best_hyperparameters_by_model.csv").exists()),
    ("4.4.2", "hyperparameter heatmaps", "4_4_2_hyperparameter_heatmap_unet.png; 4_4_2_hyperparameter_heatmap_procanet.png", (FIG_DIR / "4_4_2_hyperparameter_heatmap_unet.png").exists() and (FIG_DIR / "4_4_2_hyperparameter_heatmap_procanet.png").exists()),
    ("4.4.2", "learning rate and weight decay narrative", "4_4_2_hyperparameter_tuning_interpretation.md", (NARRATIVE_DIR / "4_4_2_hyperparameter_tuning_interpretation.md").exists()),
    ("4.4.3", "training curves and training policy", "4_4_3_training_curves.png; 4_4_3_training_policy_summary.csv", (FIG_DIR / "4_4_3_training_curves.png").exists() and (TABLE_DIR / "4_4_3_training_policy_summary.csv").exists()),
    ("4.4.3", "training stability interpretation", "4_4_3_training_stability_interpretation.md", (NARRATIVE_DIR / "4_4_3_training_stability_interpretation.md").exists()),
    ("4.4.3", "training stability summary", "4_4_3_model_stability_summary.csv", (TABLE_DIR / "4_4_3_model_stability_summary.csv").exists()),
    ("4.5", "final metrics and confusion matrix", "4_5_final_metrics.csv; 4_5_confusion_matrix_pixels.csv", True),
    ("4.6", "prediction panel and error map", "4_6_segmentation_panel_aceh_utara.png; 4_6_error_map_aceh_utara.png", True),
    ("4.7", "U-Net vs ProCANet discussion summary", "4_7_unet_vs_procanet_effectiveness_summary.csv", True),
    ("4.8", "difficult data case studies", "4_8_difficult_data_case_studies.csv", True),
    ("4.9", "BAB 4 findings and transition narrative", "4_9_bab4_findings_summary.csv; 4_7_4_9_discussion_narrative_snippets.csv", True),
]
coverage = pd.DataFrame([{"subbab": s, "todo_line_34_573_requirement": req, "artifact": art, "covered": bool(cov)} for s, req, art, cov in coverage_items])
save_table(coverage, "bab4_line_34_573_coverage_matrix.csv")
missing_coverage = coverage[~coverage["covered"]]
if missing_coverage.empty:
    print("All line 34-573 coverage items are marked covered.")
else:
    print("Coverage gaps remain:")
    display(missing_coverage)


saved table: outputs/bab4/tables/4_2_unosat_source_layer_mapping.csv (4 rows)


,layer_todo,role,notebook_artifact,status
0,FloodExtent,positive flood label,4_2_label_mask_tile_stats.csv,covered via rasterized label_flood_binary and ...
1,AnalysisExtent,valid evaluation area,4_2_label_mask_tile_stats.csv,covered via valid_mask
2,WaterExtent,permanent water exclusion/context,4_2_label_mask_tile_stats.csv,covered via water_river_mask
3,River,river mask exclusion/context,4_2_label_mask_tile_stats.csv,covered via water_river_mask


saved table: outputs/bab4/tables/4_7_4_9_discussion_narrative_snippets.csv (7 rows)


,subbab,narasi_siap_pakai
0,4.1,"Preprocessing menyatukan Sentinel-1, Sentinel-..."
1,4.2,"Label positif berasal dari FloodExtent, sedang..."
2,4.3,Validasi berbasis wilayah memisahkan area tuni...
3,4.4,U-Net dan ProCANet lolos kontrak forward pass;...
4,4.5-4.7,Perbandingan akhir perlu dibaca sebagai trade-...
5,4.8,"Wilayah dengan Sentinel-2 terbatas, water/rive..."
6,4.9,Temuan BAB 4 menjadi dasar BAB 5: kualitas inp...


saved table: outputs/bab4/tables/bab4_line_34_573_coverage_matrix.csv (15 rows)


,subbab,todo_line_34_573_requirement,artifact,covered
0,4.1.1,"raw Sentinel-1 dB, clipping [-30,0], normaliza...",4_1_1_raw_s1_db_clipping_stats.csv,True
1,4.1.1,normalized VV/VH/HSV/Slope/HAND stats and S2 v...,4_1_1_preprocessing_stats.csv,True
2,4.1.2,raster metadata/alignment,4_1_2_alignment_verification.csv,True
3,4.1.2,stack_7ch band verification,4_1_2_stack_7ch_layer_verification.csv,True
4,4.1.2,OSM overlay against stack layers,4_1_2_osm_overlay_stack_aceh_utara.png,True
5,4.2,UNOSAT source layers and label/mask stats,4_2_unosat_source_layer_mapping.csv; 4_2_label...,True
6,4.3,5-fold spatial cross-validation,4_3_five_fold_spatial_cv.csv,True
7,4.4.1,architecture and forward pass,4_4_1_model_architecture_specs.csv; 4_4_1_forw...,True
8,4.4.2,hyperparameter grid search,4_4_2_hyperparameter_tuning_summary.csv,True
9,4.4.3,training curves and training policy,4_4_3_training_curves.png; 4_4_3_training_poli...,True


All line 34-573 coverage items are marked covered.


## Checklist Output

Memverifikasi artefak wajib, sangat disarankan, dan opsional yang dibuat notebook.


In [28]:
expected = [
    ("wajib", "4.1.1", TABLE_DIR / "4_1_1_preprocessing_stats.csv"),
    ("wajib", "4.1.1", TABLE_DIR / "4_1_1_sentinel1_vv_vh_stats.csv"),
    ("wajib", "4.1.1", TABLE_DIR / "4_1_1_demnas_slope_hand_stats.csv"),
    ("wajib", "4.1.1", TABLE_DIR / "4_1_1_s2_valid_mask_by_region.csv"),
    ("wajib", "4.1.1", FIG_DIR / "4_1_1_channel_example_aceh_utara.png"),
    ("disarankan", "4.1.1", FIG_DIR / "4_1_1_s2_valid_vs_empty_comparison.png"),
    ("narasi_wajib", "4.1.1", NARRATIVE_DIR / "4_1_1_input_character_interpretation.md"),
    ("disarankan", "4.1.1", TABLE_DIR / "4_1_1_raw_s1_db_clipping_stats.csv"),
    ("disarankan", "4.1.1", TABLE_DIR / "4_1_1_preprocessing_policy_summary.csv"),
    ("wajib", "4.1.1", TABLE_DIR / "4_1_1_todo_coverage_checklist.csv"),
    ("wajib", "4.1.2", TABLE_DIR / "4_1_2_alignment_verification.csv"),
    ("disarankan", "4.1.2", TABLE_DIR / "4_1_2_stack_7ch_layer_verification.csv"),
    ("wajib", "4.1.2", FIG_DIR / "4_1_2_osm_overlay_stack_aceh_utara.png"),
    ("sangat_disarankan", "4.1.2", FIG_DIR / "4_1_2_same_tile_multilayer_panel_aceh_utara.png"),
    ("narasi_wajib", "4.1.2", NARRATIVE_DIR / "4_1_2_alignment_interpretation.md"),
    ("wajib", "4.1.2", TABLE_DIR / "4_1_2_todo_coverage_checklist.csv"),
    ("wajib", "4.2", TABLE_DIR / "4_2_label_mask_tile_stats.csv"),
    ("wajib", "4.2", FIG_DIR / "4_2_unosat_mask_panel_aceh_utara.png"),
    ("wajib", "4.2", TABLE_DIR / "4_2_unosat_mask_panel_status.csv"),
    ("wajib", "4.2", TABLE_DIR / "4_2_tile_distribution_by_split_region.csv"),
    ("wajib", "4.2", TABLE_DIR / "4_2_unosat_source_layer_mapping.csv"),
    ("narasi_wajib", "4.2", NARRATIVE_DIR / "4_2_label_tile_interpretation.md"),
    ("wajib", "4.2", TABLE_DIR / "4_2_todo_coverage_checklist.csv"),
    ("wajib", "4.3", TABLE_DIR / "4_3_spatial_split_summary.csv"),
    ("wajib", "4.3", TABLE_DIR / "4_3_five_fold_spatial_cv.csv"),
    ("wajib", "4.3", TABLE_DIR / "4_3_fold_tile_counts.csv"),
    ("disarankan", "4.3", FIG_DIR / "4_3_spatial_split_diagram.png"),
    ("narasi_wajib", "4.3", NARRATIVE_DIR / "4_3_spatial_split_interpretation.md"),
    ("wajib", "4.3", TABLE_DIR / "4_3_todo_coverage_checklist.csv"),
    ("wajib", "4.4.1", TABLE_DIR / "4_4_1_model_architecture_specs.csv"),
    ("wajib", "4.4.1", TABLE_DIR / "4_4_1_forward_pass_verification.csv"),
    ("opsional", "4.4.1", FIG_DIR / "4_4_1_unet_architecture_diagram.png"),
    ("sangat_disarankan", "4.4.1", FIG_DIR / "4_4_1_procanet_architecture_diagram.png"),
    ("narasi_wajib", "4.4.1", NARRATIVE_DIR / "4_4_1_architecture_interpretation.md"),
    ("wajib", "4.4.1", TABLE_DIR / "4_4_1_todo_coverage_checklist.csv"),
    ("wajib", "4.4.2", TABLE_DIR / "4_4_2_hyperparameter_tuning_summary.csv"),
    ("wajib", "4.4.2", TABLE_DIR / "4_4_2_grid_search_unet.csv"),
    ("wajib", "4.4.2", TABLE_DIR / "4_4_2_grid_search_procanet.csv"),
    ("wajib", "4.4.2", TABLE_DIR / "4_4_2_best_hyperparameters_by_model.csv"),
    ("disarankan", "4.4.2", FIG_DIR / "4_4_2_hyperparameter_heatmap_unet.png"),
    ("disarankan", "4.4.2", FIG_DIR / "4_4_2_hyperparameter_heatmap_procanet.png"),
    ("narasi_wajib", "4.4.2", NARRATIVE_DIR / "4_4_2_hyperparameter_tuning_interpretation.md"),
    ("wajib", "4.4.2", TABLE_DIR / "4_4_2_todo_coverage_checklist.csv"),
    ("wajib", "4.4.3", TABLE_DIR / "4_4_3_training_curve_points.csv"),
    ("wajib", "4.4.3", FIG_DIR / "4_4_3_training_curves.png"),
    ("wajib", "4.4.3", TABLE_DIR / "4_4_3_training_policy_summary.csv"),
    ("disarankan", "4.4.3", TABLE_DIR / "4_4_3_model_stability_summary.csv"),
    ("narasi_wajib", "4.4.3", NARRATIVE_DIR / "4_4_3_training_stability_interpretation.md"),
    ("wajib", "4.4.3", TABLE_DIR / "4_4_3_todo_coverage_checklist.csv"),
    ("wajib", "4.5", TABLE_DIR / "4_5_final_metrics.csv"),
    ("wajib", "4.5", TABLE_DIR / "4_5_confusion_matrix_pixels.csv"),
    ("opsional", "4.5", FIG_DIR / "4_5_fp_fn_bar_chart.png"),
    ("disarankan", "4.5", TABLE_DIR / "4_5_error_tradeoff_summary.csv"),
    ("narasi_wajib", "4.5", NARRATIVE_DIR / "4_5_final_evaluation_interpretation.md"),
    ("wajib", "4.5", TABLE_DIR / "4_5_todo_coverage_checklist.csv"),
    ("wajib", "4.6", FIG_DIR / "4_6_segmentation_panel_aceh_utara.png"),
    ("sangat_disarankan", "4.6", FIG_DIR / "4_6_error_map_aceh_utara.png"),
    ("wajib", "4.6", TABLE_DIR / "4_6_error_map_tile_counts.csv"),
    ("opsional", "4.6", TABLE_DIR / "4_6_visual_model_aspects.csv"),
    ("narasi_wajib", "4.6", NARRATIVE_DIR / "4_6_visual_spatial_interpretation.md"),
    ("wajib", "4.6", TABLE_DIR / "4_6_todo_coverage_checklist.csv"),
    ("sangat_disarankan", "4.7", TABLE_DIR / "4_7_unet_vs_procanet_effectiveness_summary.csv"),
    ("opsional", "4.7", FIG_DIR / "4_7_fp_fn_tradeoff_bar_chart.png"),
    ("opsional", "4.7", TABLE_DIR / "4_7_literature_context_comparison.csv"),
    ("narasi_wajib", "4.7", NARRATIVE_DIR / "4_7_unet_procanet_effectiveness_discussion.md"),
    ("wajib", "4.7", TABLE_DIR / "4_7_todo_coverage_checklist.csv"),
    ("sangat_disarankan", "4.8", TABLE_DIR / "4_8_difficult_data_case_studies.csv"),
    ("opsional", "4.8", FIG_DIR / "4_8_hsv_zero_tile_panel.png"),
    ("opsional", "4.8", FIG_DIR / "4_8_topography_radar_shadow_case.png"),
    ("opsional", "4.8", FIG_DIR / "4_8_permanent_water_case.png"),
    ("narasi_wajib", "4.8", NARRATIVE_DIR / "4_8_data_extreme_limitations_interpretation.md"),
    ("wajib", "4.8", TABLE_DIR / "4_8_todo_coverage_checklist.csv"),
    ("opsional", "4.9", TABLE_DIR / "4_9_bab4_findings_summary.csv"),
]
checklist = pd.DataFrame([
    {"priority": prio, "subbab": subbab, "artifact": str(path.relative_to(ROOT)), "exists": path.exists(), "size_bytes": path.stat().st_size if path.exists() else 0}
    for prio, subbab, path in expected
])
save_table(checklist, "bab4_output_checklist.csv")
missing = checklist[~checklist["exists"]]
if not missing.empty:
    print("Missing expected artifacts:")
    display(missing)
else:
    print("All expected BAB 4 artifacts created.")


saved table: outputs/bab4/tables/bab4_output_checklist.csv (21 rows)


,priority,subbab,artifact,exists,size_bytes
0,wajib,4.1.1,outputs/bab4/tables/4_1_1_preprocessing_stats.csv,True,8929
1,wajib,4.1.1,outputs/bab4/tables/4_1_1_sentinel1_vv_vh_stat...,True,2840
2,wajib,4.1.1,outputs/bab4/tables/4_1_1_demnas_slope_hand_st...,True,2666
3,wajib,4.1.1,outputs/bab4/tables/4_1_1_s2_valid_mask_by_reg...,True,393
4,wajib,4.1.1,outputs/bab4/figures/4_1_1_channel_example_ace...,True,6190022
5,disarankan,4.1.1,outputs/bab4/figures/4_1_1_s2_valid_vs_empty_c...,True,2036582
6,narasi_wajib,4.1.1,outputs/bab4/narratives/4_1_1_input_character_...,True,1099
7,wajib,4.1.2,outputs/bab4/tables/4_1_2_alignment_verificati...,True,223
8,wajib,4.2,outputs/bab4/tables/4_2_label_mask_tile_stats.csv,True,1260
9,wajib,4.2,outputs/bab4/tables/4_2_tile_distribution_by_s...,True,1262


All expected BAB 4 artifacts created.
